# Análise PCA e MDS com dados da ANP

Este notebook registra o caminho usado pelo grupo para estudar a relação entre preço da gasolina C, volume vendido e participação do etanol hidratado. A ideia é manter a análise próxima do problema do projeto: uma rede de postos ou distribuidora precisa entender melhor diferenças regionais antes de decidir estoque, mix comercial e campanhas.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))
import matplotlib.pyplot as plt
import pandas as pd
from src.data_preparation import FEATURES_NUMERICAS, padronizar_features, preparar_dados
from src.pca_analysis import aplicar_pca
from src.mds_analysis import aplicar_mds

## Preparação dos dados

As bases da ANP vêm separadas: uma traz preços médios por estado e outra traz volumes vendidos por produto. Nesta etapa, os nomes de estados são normalizados, gasolina C e etanol hidratado são selecionados, e as tabelas são cruzadas por mês e UF (a sigla já determina a região; o rótulo de região é normalizado entre as fontes, por exemplo equivalendo "Centro Oeste" a "Centro-Oeste"). Em seguida são calculadas variáveis derivadas para comparar preço, volume e participação do etanol.

In [ ]:
dados = preparar_dados(periodo_inicio=2021, periodo_fim=2025)
dados_padronizados, scaler = padronizar_features(dados)
dados.shape, FEATURES_NUMERICAS

## Conferência inicial

Antes de aplicar PCA e MDS, vale olhar algumas linhas do dataset tratado. Cada registro representa uma combinação de mês e UF. As colunas de variação ajudam a observar mudanças mensais, enquanto participação e razão do etanol mostram o peso do combustível substituto.

In [ ]:
dados[['mes_ano', 'uf', 'regiao', 'preco_medio_gasolina_c', 'volume_gasolina_c_m3', 'volume_etanol_hidratado_m3', 'participacao_etanol']].head()

## Padronização

A padronização evita que variáveis de volume, que têm valores muito maiores, dominem a análise. Depois desse passo, cada feature numérica passa a ser comparada em uma escala comum.

In [ ]:
dados_padronizados.describe().round(3)

## PCA

O PCA cria novas dimensões que concentram a variação dos dados. Aqui usamos duas componentes para visualizar os registros em um plano. Também analisamos as cargas de cada variável, pois elas indicam quais features mais pesam em cada componente.

In [ ]:
pca_df, cargas, variancia, pca = aplicar_pca(dados_padronizados)
display(variancia)
display(cargas.sort_values('PC1', key=abs, ascending=False))

## Gráfico do PCA

No gráfico, cada ponto é um registro UF-mês. A cor por região ajuda a perceber se há separação regional. Pontos muito afastados merecem atenção porque podem representar mercados de escala diferente, maior participação do etanol ou meses com mudança brusca.

In [ ]:
pca_plot = dados.join(pca_df)
ax = pca_plot.plot.scatter(x='PC1', y='PC2', c='participacao_etanol', colormap='viridis', figsize=(8, 6))
ax.set_title('PCA 2D por participação do etanol')
ax.set_xlabel(f"PC1 ({variancia.loc[0, 'variancia_explicada']:.1%})")
ax.set_ylabel(f"PC2 ({variancia.loc[1, 'variancia_explicada']:.1%})")
plt.show()

## MDS

O MDS parte das distâncias entre registros. Ele não mostra diretamente quais variáveis explicam os eixos, mas ajuda a enxergar quais meses e estados ficaram próximos no perfil geral de preço, volume e participação do etanol.

In [ ]:
mds_df, mds = aplicar_mds(dados_padronizados, max_registros=800)
print(f'Stress MDS: {mds.stress_:.2f}')
mds_df.head()

## Gráfico do MDS

Nesta visualização, pontos próximos indicam registros com comportamento parecido nas features padronizadas. A leitura deve ser feita com cuidado, pois a projeção em duas dimensões simplifica relações que originalmente estão em mais variáveis.

In [ ]:
mds_plot = dados.loc[mds_df.index].join(mds_df)
ax = mds_plot.plot.scatter(x='MDS1', y='MDS2', c='participacao_etanol', colormap='plasma', figsize=(8, 6))
ax.set_title('MDS 2D por participação do etanol')
ax.set_xlabel('MDS1')
ax.set_ylabel('MDS2')
plt.show()

## Leitura final

O PCA foi mais útil para explicar a influência das variáveis. O MDS complementou a análise ao mostrar proximidade entre registros. Em conjunto, os dois métodos indicam que o comportamento do mercado depende de preço, escala de vendas e presença do etanol, não apenas de uma variável isolada.